# Solution E3 · The scaler refitted on the test set

**Type**: Find the bug

## Objective

Find out why a model with excellent training results has a test error ten times larger, fix
the preprocessing and verify that a `Pipeline` produces exactly the same correct result.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = ds.trending_series()
X, y = d[["x1", "x2"]], d["y"]
cut = int(0.75 * len(d))
X_tr, X_te, y_tr, y_te = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]

## Commented solution

**Where the bug is.** `scaler.fit_transform(X_te)` recomputes mean and standard deviation **on the test set** and
re-centers it. But the model was trained on features centered with respect to the training set: in the last 100 rows the
features have risen (trend), so in "training" units they are worth about +2 standard deviations. By re-centering the test,
the model sees values around 0 and predicts as if we were back at the average level of the past.

In [ ]:
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
model = LinearRegression().fit(X_tr_s, y_tr)

# what the model should see on the test set
test_mean_std_right = scaler.transform(X_te).mean(axis=0)
print("test means, in training standard deviations:", test_mean_std_right.round(2))

In [ ]:
X_te_bug = StandardScaler().fit_transform(X_te)              # the colleague's code
rmse_bug = mean_squared_error(y_te, model.predict(X_te_bug), squared=False)

X_te_ok = scaler.transform(X_te)                             # transform only, with the training statistics
pred_manual = model.predict(X_te_ok)
rmse_ok = mean_squared_error(y_te, pred_manual, squared=False)
print(f"test RMSE: with the bug {rmse_bug:.2f} | fixed {rmse_ok:.2f}")

In [ ]:
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
pred_pipe = pipe.predict(X_te)
print("the Pipeline matches the correct manual preprocessing:", np.allclose(pred_pipe, pred_manual))

## Check

In [ ]:
with verify():
    check(rmse_bug > 3 * rmse_ok, "the wrong preprocessing makes the error more than 3 times worse",
          "rmse_ok must come from X_te standardized with scaler.transform (training statistics)")
    check(np.all(np.abs(np.asarray(test_mean_std_right)) > 1),
          "in the test, the features are more than one standard deviation above the training level",
          "test_mean_std_right = scaler.transform(X_te).mean(axis=0): the trend takes the features far above the training means")
    check(close(pred_manual, model.predict(scaler.transform(X_te))), "pred_manual uses the scaler fitted on the training set",
          "pred_manual must be model.predict(scaler.transform(X_te))")
    check(close(pred_pipe, pred_manual), "the Pipeline produces exactly the same predictions",
          "the Pipeline trained on X_tr must give the same predictions as the correct manual preprocessing")
    check(rmse_ok < 2, "the fixed RMSE is close to the noise of the target", "rmse_ok should be about 1")
done()

## What you learned

- `fit` (computes the statistics) is done **only** on the training set; `transform` (applies them) is used on training, validation
  and test with the same statistics. `fit_transform(X_te)` is almost always a bug.
- Here the bug does not make you "cheat" (the test is not at an advantage) but it **breaks** the model: it erases the trend and produces a
  test error of about 8 against 0.9. With non-stationary data (with a drift) it is the normal consequence, not a
  strange case.
- A `Pipeline` makes the bug impossible by construction: `fit` on the training set calls `fit_transform` of the intermediate
  steps, `predict` calls only `transform`.

## Common pitfall

Standardizing "per fold" with `fit_transform` on every validation fold (it happens when you write the
cross-validation by hand). The scaler must be fitted on the training part of the fold and applied to the validation: it is exactly what
`cross_val_score` does with a Pipeline.

## Interview question

> *Click on a question to show its answer.*

<details><summary>In production your model receives one request at a time. How do you apply the scaler to a single request?</summary>

With `transform`, using the mean and standard deviation saved in training: it is also the reason why the scaler object
(or better the whole Pipeline) must be serialized with the model. With a single row `fit_transform` would give a standard
deviation of zero and would center the value on itself, that is no information.

</details>